# 캡스톤 MCP 문서 서버

- 구분: 정답 예시
- 원본: [`mcp_server.py`](mcp_server.py)
- 연결 교안: [4-3-research-automation-multi-agent-system.pdf](../../../../docs/raw/Chapter02/day02/4-3-research-automation-multi-agent-system.pdf)

원본 Python 코드는 수정 없이 아래의 한 코드 셀에 담았습니다. 설명과 실행 준비는 별도 셀입니다. **모두 실행** 대신 필요한 셀을 직접 선택하세요.


## 학습 목표

1. MCP 도구의 입력·출력과 SSE 서버 주소를 설명한다.
2. 코드를 실행하기 전 입력, 출력, 외부 호출과 파일 변경 가능성을 설명한다.
3. CodeMate가 만든 결과를 원본과 교안에 비춰 검토한다.


## 코드를 읽는 순서

DATA_DIR → list_docs → read_doc → search_docs → mcp.run 순서로 읽는다.

교안은 개념과 목표를 설명합니다. 코드에서 실제 구현된 부분과 TODO를 구분하세요.


## 실행 준비

저장소 안에서 VS Code로 열고 프로젝트 `.venv` 커널을 선택합니다. 의존성은 `uv sync`로 준비합니다. 사내망에서는 사내 LLM 환경을, 사외에서는 허용된 provider를 준비합니다. 자격증명 값은 셀에 적지 않습니다. 아래 준비 셀은 저장소 루트를 찾고 `__file__`을 원본 Python 파일 경로로 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys
_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents) if (p / "pyproject.toml").is_file() and (p / "config" / "llm.py").is_file()), None)
if _repo is None:
    raise FileNotFoundError("저장소 안에서 노트북을 열어 주세요.")
os.chdir(_repo)
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
__file__ = str(_repo / 'src/solution/Chapter02/06-capstone/mcp_server.py')
import __main__
sys.modules.setdefault("notebook_source", __main__)
__name__ = "notebook_source"  # 스크립트 전용 실행 부분은 아래에서 따로 실행
print(f"실행 기준 폴더: {_repo}")


## 원본 코드 — 직접 실행

**조건과 부작용:** `0.0.0.0:8001` SSE 서버가 계속 실행되어 셀을 점유한다. `read_doc`은 DATA_DIR 밖 경로를 막지 않으므로 신뢰하지 않는 파일명을 전달하지 않는다. 이 파일은 정답본에만 있다.

준비 셀을 먼저 실행하고 필요한 서비스·설정을 확인하세요. 다음 셀은 `manual-run`으로 표시했습니다. 원본 전체가 한 번에 실행되므로 API 요청, 파일 변경 또는 대기가 생길 수 있습니다.


In [ ]:
"""
캡스톤 MCP 파일시스템 서버

data/ 폴더의 문서를 Researcher 에이전트가 검색할 수 있도록 MCP 도구로 제공한다.

실행 (solution.py 실행 전 먼저 실행):
  python src/solution/Chapter02/06-capstone/mcp_server.py
"""

import os
from pathlib import Path

from fastmcp import FastMCP

DATA_DIR = Path(__file__).parent / 'data'

mcp = FastMCP('capstone-filesystem')


@mcp.tool()
def list_docs() -> list[str]:
    """data/ 폴더에 있는 문서 목록을 반환한다."""
    return [f.name for f in DATA_DIR.glob('*.md')]


@mcp.tool()
def read_doc(filename: str) -> str:
    """data/ 폴더의 문서를 읽어 전체 내용을 반환한다."""
    path = DATA_DIR / filename
    if not path.exists():
        return f'파일 없음: {filename}'
    return path.read_text(encoding='utf-8')


@mcp.tool()
def search_docs(keyword: str) -> str:
    """data/ 폴더의 모든 문서에서 키워드를 검색해 관련 단락을 반환한다."""
    results = []
    for doc_path in DATA_DIR.glob('*.md'):
        content = doc_path.read_text(encoding='utf-8')
        # 키워드가 포함된 단락만 추출
        paragraphs = [p.strip() for p in content.split('\n\n') if keyword in p]
        if paragraphs:
            results.append(f'[{doc_path.name}]\n' + '\n'.join(paragraphs[:3]))
    return '\n\n'.join(results) if results else f'"{keyword}" 검색 결과 없음'


if __name__ == '__main__':
    # SSE 모드로 실행 — solution.py가 http://localhost:8001/sse 로 연결
    mcp.run(transport='sse', host='0.0.0.0', port=8001)


## 서버 시작

권장 실행은 별도 터미널의 `python3 src/solution/Chapter02/06-capstone/mcp_server.py`입니다. 아래 셀도 서버를 시작하지만 커널을 점유하고 포트 8001을 열므로 직접 선택할 때만 실행합니다.


In [ ]:
mcp.run(transport='sse', host='0.0.0.0', port=8001)


## CodeMate에게 요청하고 검토하기

> 세 도구의 파일 접근 범위와 입력 검증을 표로 정리하고 클라이언트의 SSE 주소와 비교해 줘.

CodeMate가 제안한 수정의 근거를 읽고, 실행 가능한 환경에서 출력·Trace·생성 파일을 확인하세요.


## 확인 질문

1. 서버가 찾는 확장자는? `read_doc`에 경로 문자열이 들어오면? 8001 포트가 열리면?
2. 실행 전 어떤 환경과 자료를 확인해야 하는가?
3. 결과가 기대와 다르면 어느 단계의 기록부터 볼 것인가?
